# fullstack — Databricks notebook (PySpark + Scala)

**Part 1 — PySpark:** sample customers → Parquet → token columns → Parquet via `fullstack.jobs.email_tokenization`.

**Part 2 — Scala:** the same transformation in a **`%scala`** cell using the **shared cluster `spark`** (mirrors [`TokenizeEmailsJob.scala`](../scala/tokenize-email-job/src/main/scala/fullstack/jobs/TokenizeEmailsJob.scala)).

**Local Jupyter:** `%scala` is **Databricks-only**. Run through Part 1 locally with `poetry install` + Jupyter; skip Part 2 or use [`fullstack_local_spark.ipynb`](fullstack_local_spark.ipynb).

**Import:** Upload or open from **Databricks Repos**.

**Setup:** Wheel via **`pip_wheel_uri`**, cluster library, or `%pip`. Two tokenized URIs (**`tokenized_parquet_uri`** vs **`tokenized_scala_parquet_uri`**) avoid PySpark overwriting Scala output during the demo.

Do not call **`spark.stop()`** on a shared cluster.


In [ ]:
import os
import subprocess
import sys

IS_DATABRICKS = bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))

if IS_DATABRICKS:
    dbutils.widgets.text("customers_parquet_uri", "dbfs:/tmp/fullstack/customers_parquet")
    dbutils.widgets.text("tokenized_parquet_uri", "dbfs:/tmp/fullstack/customers_tokenized")
    dbutils.widgets.text(
        "tokenized_scala_parquet_uri",
        "dbfs:/tmp/fullstack/customers_tokenized_scala",
    )
    dbutils.widgets.text("pip_wheel_uri", "")


## Optional: install the `fullstack` package

Set **`pip_wheel_uri`** to a uploaded wheel path (workspace or Volume), or leave empty if `fullstack` is already installed.


In [ ]:
if IS_DATABRICKS:
    wheel = dbutils.widgets.get("pip_wheel_uri").strip()
    if wheel:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", wheel]
        )
    else:
        print("pip_wheel_uri empty — assuming `fullstack` is installed.")
else:
    print("Local: run from repo with `poetry install`.")


In [ ]:
import tempfile
from pathlib import Path


def rstrip_slash(uri: str) -> str:
    return uri.rstrip("/")


if IS_DATABRICKS:
    CUSTOMERS_URI = rstrip_slash(dbutils.widgets.get("customers_parquet_uri"))
    TOKENIZED_URI = rstrip_slash(dbutils.widgets.get("tokenized_parquet_uri"))
    TOKENIZED_SCALA_URI = rstrip_slash(
        dbutils.widgets.get("tokenized_scala_parquet_uri")
    )
else:
    base = Path(tempfile.gettempdir()) / "fullstack_nb_demo"
    base.mkdir(parents=True, exist_ok=True)
    CUSTOMERS_URI = str(base / "customers_parquet")
    TOKENIZED_URI = str(base / "customers_tokenized")
    TOKENIZED_SCALA_URI = str(base / "customers_tokenized_scala")

print("Customers Parquet URI:", CUSTOMERS_URI)
print("Tokenized (PySpark) URI:", TOKENIZED_URI)
print("Tokenized (Scala) URI:", TOKENIZED_SCALA_URI)


In [ ]:
if not IS_DATABRICKS:
    import sys
    from pathlib import Path

    here = Path.cwd()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").is_file():
            src = cand / "src"
            if str(src) not in sys.path:
                sys.path.insert(0, str(src))
            break
    else:
        raise RuntimeError(
            "Could not find pyproject.toml; run this notebook from the fullstack repo."
        )


## Part 1 — PySpark


In [ ]:
from pyspark.sql import functions as F

from fullstack.jobs.email_tokenization import with_email_token_columns
from fullstack.spark_session import get_spark

spark = get_spark(app_name="fullstack-databricks-notebook")
print("Spark:", spark.version)


In [ ]:
customers_df = spark.createDataFrame(
    [
        (1, "Alice", "alice@example.com"),
        (2, "Bob", "bob@example.com"),
        (3, "Carlos", "carlos@example.com"),
    ],
    ["customer_id", "name", "email"],
)

customers_enriched = customers_df.withColumn(
    "ingested_at",
    F.to_timestamp(F.lit("2026-01-01 09:30:00")),
).repartition(2)

customers_enriched.show(truncate=False)
customers_enriched.write.mode("overwrite").parquet(CUSTOMERS_URI)


In [ ]:
read_customers = spark.read.parquet(CUSTOMERS_URI)
tokenized = with_email_token_columns(read_customers)
tokenized.show(truncate=False)
tokenized.printSchema()
tokenized.write.mode("overwrite").parquet(TOKENIZED_URI)


In [ ]:
spark.read.parquet(TOKENIZED_URI).orderBy("customer_id").show(truncate=False)


## Part 2 — Scala (`%%scala`)

Runs **only on Databricks** (not standard Jupyter). Uses the notebook-bound **`spark`**; no `SparkSession.builder` / `spark.stop()` here.


In [ ]:
%%scala
import org.apache.spark.sql.SaveMode
import org.apache.spark.sql.functions._

def uri(s: String): String =
  if (s.endsWith("/")) s.substring(0, s.length - 1) else s

val customersPath = uri(dbutils.widgets.get("customers_parquet_uri"))
val tokenizedScalaPath = uri(dbutils.widgets.get("tokenized_scala_parquet_uri"))

val df = spark.read.parquet(customersPath)
val emailCol = trim(col("email"))
val atParts = split(emailCol, "@")
val localPart = element_at(atParts, 1)
val domainPart = element_at(atParts, 2)
val tokenized = df
  .withColumn("email_local", localPart)
  .withColumn("email_domain", domainPart)
  .withColumn("email_local_tokens", split(localPart, "\\."))
  .withColumn("email_domain_tokens", split(domainPart, "\\."))
  .withColumn(
    "email_token_string",
    concat_ws(
      " @ ",
      concat_ws(".", col("email_local_tokens")),
      concat_ws(".", col("email_domain_tokens"))))

tokenized.show(false)
tokenized.printSchema()
tokenized.write.mode(SaveMode.Overwrite).parquet(tokenizedScalaPath)


In [ ]:
if IS_DATABRICKS:
    # Assumes Part 2 %%scala cell ran and wrote TOKENIZED_SCALA_URI
    cols = sorted(
        set(spark.read.parquet(TOKENIZED_URI).columns)
        & set(spark.read.parquet(TOKENIZED_SCALA_URI).columns)
    )
    df_py = spark.read.parquet(TOKENIZED_URI).select(*cols).orderBy("customer_id")
    df_scala = spark.read.parquet(TOKENIZED_SCALA_URI).select(*cols).orderBy(
        "customer_id"
    )
    n_py, n_scala = df_py.count(), df_scala.count()
    d1 = df_py.subtract(df_scala).count()
    d2 = df_scala.subtract(df_py).count()
    print(f"Rows PySpark={n_py} Scala={n_scala}")
    print(
        "Subtract asymmetry rows:", d1 + d2, "(expect 0 when both paths match)"
    )
else:
    print(
        "Compare skipped locally: run Part 2 on Databricks after the `%%scala` cell."
    )
